# 02 - Continual learning with replay and incremental polynomial expansion

Experiences are processed one at a time. At experience *t* the learner may use only the current experience's training
samples, its replay buffer (training samples of earlier experiences), and its current parameters. Evaluation samples are
never used for learning or replay. The polynomial degree grows by one per experience (default), following the **expansion
invariant**: old coefficients are copied exactly, new coefficients start at zero, and `f_{d+1}(x) = f_d(x)` is verified (scores and
probabilities, on training, evaluation, random and out-of-region inputs) before any training.

Nothing here assumes the method works. Compare the strategies at the end before drawing any conclusion.

## Configuration

In [ ]:
SEED = 0
N_TRAIN, N_TEST, NOISE = 200, 500, 0.0
STRATEGY = "ordinary"  # ordinary_no_replay | ordinary | frozen_old | small_old_lr | strategy_b_all | strategy_b_new
STOP_AFTER = (
    None  # e.g. 3 stops after experience 3 so intermediate behaviour can be inspected
)
DEGREES = None  # None -> 1, 2, ..., 10 ; or give an explicit non-decreasing list
LR, EPOCHS = 0.5, 300
M_CURRENT, M_REPLAY, CAPACITY, REPLACE = (
    100,
    100,
    100,
    False,
)  # balanced replay: m per experience
SHOW_STEPS = [
    3,
    6,
    10,
]  # experiences (beyond 1 and 2, shown above) for which the before/after figure is drawn

In [ ]:
import pathlib
import sys

# Resolve the repository root from either the repository root or notebooks/ cwd.
_cwd = pathlib.Path.cwd().resolve()
_candidates = (_cwd, *_cwd.parents)
_REPO_ROOT = next(
    (
        candidate for candidate in _candidates
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "src" / "polynomial_cl").is_dir()
    ),
    None,
)
if _REPO_ROOT is None:
    raise RuntimeError(
        "Could not locate the repository root. Start Jupyter from the repository "
        "root or its notebooks/ directory."
    )

# Anchor repository and local data paths, independent of the notebook kernel working directory.
for _import_path in (_REPO_ROOT, _REPO_ROOT / "src"):
    if str(_import_path) not in sys.path:
        sys.path.insert(0, str(_import_path))

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams.update({"figure.dpi": 80})
from polynomial_cl import (
    STRATEGIES,
    make_config,
    make_experiences,
    metrics,
    run_continual,
    run_seeds,
)
from polynomial_cl import plotting as P

In [ ]:
exps = make_experiences(seed=SEED, n_train=N_TRAIN, n_test=N_TEST, noise=NOISE)
cfg = make_config(
    STRATEGY,
    lr=LR,
    epochs=EPOCHS,
    m_current=M_CURRENT,
    m_replay=M_REPLAY,
    capacity=CAPACITY,
    replace=REPLACE,
    seed=SEED,
    stop_after=STOP_AFTER,
    degrees=DEGREES,
)
fig, axs = plt.subplots(1, 2, figsize=(9, 4.2))
P.plot_ground_truth(axs[0], title="Ground truth")
P.plot_ground_truth(axs[1], title="Experiences = vertical strips (training samples)")
for e in exps:
    P.plot_samples(axs[1], e.train.x, e.train.y, s=6)
    axs[1].axvline(e.window[0], color="k", lw=0.3)
plt.tight_layout()
plt.show()
result = run_continual(exps, cfg)
print(cfg)

## Stage A - initial model (experience 1)

In [ ]:
r1 = result.records[0]
print("degree:", r1.degree_after, "| features:", r1.feature_names)
print(f"training loss: {r1.loss_history[0]:.4f} -> {r1.loss_history[-1]:.4f}")
print(f"accuracy on experience 1 (fixed test set): {r1.acc_current:.3f}")
stage_a = {
    "scores": r1.model_after.score(exps[0].test.x),
    "labels": r1.model_after.predict(exps[0].test.x),
    "coefficients": r1.model_after.w.copy(),
}  # saved for later comparison
print(P.format_coefficient_table(r1))
fig = P.plot_step(r1, exps)
plt.show()

## Stage B - experience 2: expand, verify the invariant, then train

The second panel (right after expansion, no training) must be identical to the first.

In [ ]:
r2 = result.records[1]
print(
    f"expansion degree {r2.degree_before} -> {r2.degree_after}:  max|score diff| = {r2.invariant_score_diff:.2e},  max|proba diff| = {r2.invariant_proba_diff:.2e}"
)
print("replay batch composition (samples drawn per experience):", r2.batch_counts)
print(P.format_coefficient_table(r2))
P.plot_step(r2, exps)
plt.show()
P.plot_coefficients(r2)
plt.show()

## Stage C - experiences 3 ... 10

In [ ]:
for t in SHOW_STEPS:
    if t - 1 < len(result.records):
        r = result.records[t - 1]
        print(
            f"--- experience {t}: degree {r.degree_before} -> {r.degree_after}, batch {r.batch_counts}"
        )
        print(
            f"    invariant: score diff {r.invariant_score_diff:.1e}, proba diff {r.invariant_proba_diff:.1e}"
        )
        P.plot_step(r, exps)
        plt.show()
        P.plot_coefficients(r)
        plt.show()

## Learning history and compact summary

In [ ]:
P.plot_learning_history(result)
plt.show()
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
P.plot_accuracy_matrix(result, axs[0])
last = result.records[-1].model_after
x_te = np.concatenate([e.test.x for e in exps[: len(result.records)]])
y_te = np.concatenate([e.test.y for e in exps[: len(result.records)]])
P.plot_correctness(
    last,
    axs[1],
    samples=(x_te, y_te),
    title="Final model: correct/incorrect regions + all evaluation samples",
)
plt.tight_layout()
plt.show()
print(metrics.format_table(metrics.summary_table(result)))

## Strategy comparison (single seed, identical data, evaluation sets, learning rate, epochs and replay budget)

| name | what changes |
|---|---|
| `ordinary_no_replay` | one global learning rate, **no replay** (isolates the effect of replay) |
| `ordinary` | one global learning rate, replay |
| `frozen_old` | existing coefficients frozen; only newly introduced coefficients train |
| `small_old_lr` | existing coefficients use 0.1 x the learning rate |
| `strategy_b_all` | `f = f_old + g` with the constraint `g(x_ref) = 0` on reference inputs (3 buffered inputs per old experience); g over all coefficients |
| `strategy_b_new` | as above, g only over new coefficients |

Preserving `f_old` at a finite set of reference points does **not** preserve it elsewhere; the columns below measure that directly.

In [ ]:
names = list(STRATEGIES)

def cfg_for(n, **kw):
    shared = {
        "lr": LR,
        "epochs": EPOCHS,
        "m_current": M_CURRENT,
        "m_replay": M_REPLAY,
        "capacity": CAPACITY,
        "replace": REPLACE,
    }
    # A preset owns its defining parameters; shared settings only fill the rest.
    shared = {k: v for k, v in shared.items() if k not in STRATEGIES[n]}
    return make_config(n, **shared, **kw)

runs = {n: run_continual(exps, cfg_for(n, seed=SEED)) for n in names}
rows = []
for n, r in runs.items():
    last = r.records[-1]
    rows.append({
        "strategy": n,
        "final_mean_acc": last.mean_acc,
        "final_acc_current": last.acc_current,
        "final_acc_E1": r.acc_matrix[-1, 0],
        "final_mean_forgetting": last.mean_forgetting,
        "final_train_loss": float(last.loss_history[-1]),
        "loss_reduction_last_exp": float(last.loss_history[0] - last.loss_history[-1]),
        "trainable_dims_last_exp": last.n_trainable,
        "trainable_l1_update_last_exp": last.trainable_l1_update,
        "max_old_score_change_last_exp": last.max_old_score_change,
        "old_flip_fraction_last_exp": last.old_flip_fraction,
        # null_dim is meaningful only for Strategy B's constrained update.
        "strategy_b_null_dim": last.null_dim if last.null_dim is not None else "n/a",
        "max_frozen_change": max(
            (x.frozen_max_change for x in r.records if x.frozen_max_change is not None),
            default=float("nan"),
        ),
    })
print(metrics.format_table(rows))

fig, ax = plt.subplots(figsize=(8, 4.5))
for n, r in runs.items():
    ax.plot(range(1, len(r.records) + 1), [x.mean_acc for x in r.records],
            marker="o", ms=3, label=n)
ax.set_xlabel("Experience")
ax.set_ylabel("Mean accuracy on observed experiences")
ax.set_title("Strategy comparison: accuracy and learning behavior")
ax.legend(fontsize=7)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for n, r in runs.items():
    axes[0].plot(range(1, len(r.records) + 1),
                 [float(x.loss_history[0] - x.loss_history[-1]) for x in r.records],
                 marker="o", ms=3, label=n)
    axes[1].plot(range(1, len(r.records) + 1),
                 [x.trainable_l1_update for x in r.records],
                 marker="o", ms=3, label=n)
axes[0].set_title("Loss reduction within each experience")
axes[0].set_xlabel("Experience")
axes[0].set_ylabel("Initial loss − final loss")
axes[1].set_title("Magnitude of trainable parameter updates")
axes[1].set_xlabel("Experience")
axes[1].set_ylabel("L1 update")
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

### Ablation: does the incremental degree schedule matter?

Same data, replay and budget, but the model has degree 10 from experience 1 (all 66 coefficients exist and are trainable
from the start) instead of growing 1, 2, ..., 10. Five seeds.

In [ ]:
# Paired-seed ablation: does expansion help versus no expansion, and does timing matter?
# All conditions reuse the same generated experience stream within each seed.
abl = {}
degree_conditions = (
    ("fixed degree 1 (no expansion)", [1] * len(exps)),
    ("grow 1..10", None),
    ("fixed degree 10 (capacity control)", [10] * len(exps)),
)
for label, deg in degree_conditions:
    rs = run_seeds(
        [0, 1, 2, 3, 4],
        cfg_for("ordinary", degrees=deg),
        {"n_train": N_TRAIN, "n_test": N_TEST, "noise": NOISE},
    )
    abl[label] = rs
    final_acc = np.asarray([r.records[-1].mean_acc for r in rs])
    final_forgetting = np.asarray([r.records[-1].mean_forgetting for r in rs])
    print(
        f"{label:34s} "
        f"final mean accuracy {final_acc.mean():.3f} +/- {final_acc.std(ddof=1):.3f} | "
        f"final mean forgetting {final_forgetting.mean():.3f} +/- "
        f"{final_forgetting.std(ddof=1):.3f}"
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for label, rs in abl.items():
    per_seed_acc = np.asarray([[step.mean_acc for step in r.records] for r in rs])
    per_seed_forgetting = np.asarray([[step.mean_forgetting for step in r.records] for r in rs])
    x = np.arange(1, per_seed_acc.shape[1] + 1)
    axes[0].plot(x, per_seed_acc.mean(axis=0), marker="o", ms=3, label=label)
    axes[0].fill_between(x, per_seed_acc.mean(axis=0) - per_seed_acc.std(axis=0, ddof=1),
                         per_seed_acc.mean(axis=0) + per_seed_acc.std(axis=0, ddof=1), alpha=0.15)
    axes[1].plot(x, per_seed_forgetting.mean(axis=0), marker="o", ms=3, label=label)
axes[0].set_title("Mean accuracy across experiences")
axes[0].set_xlabel("Experience")
axes[0].set_ylabel("Mean accuracy")
axes[1].set_title("Mean forgetting across experiences")
axes[1].set_xlabel("Experience")
axes[1].set_ylabel("Mean forgetting")
axes[0].legend(fontsize=7)
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("- expanding vs fixed degree 1 tests whether adding polynomial capacity helps at all.")
print("- expanding vs fixed degree 10 tests whether gradual availability helps versus final capacity from the start.")

### Strategy B inspection

`ref_score_change` (change at the reference inputs) should be ~1e-15 by construction. `old_score_change` is measured on the
*whole* old evaluation sets, which are not the reference inputs. `free dims` is specific to Strategy B and is not a general learning indicator. For every strategy, inspect the loss history, `loss_reduction`, and `trainable_l1_update`; `null_dim=None` means no null-space constraint was applied, not that learning did not happen. A zero free dimension in Strategy B means g cannot update.

In [ ]:
for n in ("strategy_b_all", "strategy_b_new"):
    print("==", n)
    print(
        metrics.format_table(
            [
                {
                    "exp": r.t + 1,
                    "degree": r.degree_after,
                    "null_dim": r.null_dim if r.null_dim is not None else "n/a (unconstrained)",
                    "trainable_dims": r.n_trainable,
                    "loss_reduction": float(r.loss_history[0] - r.loss_history[-1]),
                    "trainable_l1_update": r.trainable_l1_update,
                    "acc_current": r.acc_current,
                    "mean_acc": r.mean_acc,
                    "ref_score_change": r.ref_score_change
                    if r.ref_score_change is not None else float("nan"),
                    "old_score_change(eval sets)": r.max_old_score_change,
                    "old_flip_frac": r.old_flip_fraction,
                    "frozen_change": r.frozen_max_change
                    if r.frozen_max_change is not None else float("nan"),
                }
                for r in runs[n].records
            ],
            "{:.3g}",
        )
    )
    P.plot_learning_history(runs[n])
    plt.show()
    P.plot_step(runs[n].records[-1], exps)
    plt.show()

print("A null dimension is a constraint diagnostic, not a general learning indicator.")
print("For unconstrained strategies, inspect loss reduction, trainable updates, accuracy, and forgetting instead.")

# Compare the final binary decision boundaries on the same domain and test samples.
# "ordinary" is the distinct preset equivalent to the former unconstrained Strategy A-all.
comparison_names = (
    ("ordinary (former A-all behavior)", "ordinary"),
    ("frozen_old (former A-new behavior)", "frozen_old"),
    ("Strategy B: all coefficients", "strategy_b_all"),
    ("Strategy B: new coefficients", "strategy_b_new"),
)
x_eval = np.concatenate([e.test.x for e in exps[:len(result.records)]])
y_eval = np.concatenate([e.test.y for e in exps[:len(result.records)]])
fig, axes = plt.subplots(1, len(comparison_names), figsize=(20, 4.5))
for ax, (label, strategy_name) in zip(axes, comparison_names):
    P.plot_correctness(
        runs[strategy_name].records[-1].model_after,
        ax=ax,
        samples=(x_eval, y_eval),
        title=label,
        legend=False,
    )
plt.tight_layout()
plt.show()


## Multiple seeds

Seeds change the training samples, replay selection and shuffling (evaluation sets also change with the seed but are
fixed within a seed). Values are mean +/- standard deviation over seeds.

In [ ]:
SEEDS = [0, 1, 2, 3, 4]


def mean_std(runs_, getter):
    vals = [getter(r) for r in runs_]
    return np.mean(vals), np.std(vals)


rows = []
for n in names:
    base = cfg_for(n)
    rs = run_seeds(SEEDS, base, {"n_train": N_TRAIN, "n_test": N_TEST, "noise": NOISE})
    m1, s1 = mean_std(rs, lambda r: r.records[-1].mean_acc)
    m2, s2 = mean_std(rs, lambda r: r.records[-1].acc_current)
    m3, s3 = mean_std(rs, lambda r: r.records[-1].mean_forgetting)
    m4, s4 = mean_std(rs, lambda r: r.acc_matrix[-1, 0])
    rows.append(
        {
            "strategy": n,
            "final_mean_acc": f"{m1:.3f}+/-{s1:.3f}",
            "final_acc_current": f"{m2:.3f}+/-{s2:.3f}",
            "final_acc_E1": f"{m4:.3f}+/-{s4:.3f}",
            "final_mean_forgetting": f"{m3:.3f}+/-{s3:.3f}",
        }
    )
print(metrics.format_table(rows))

## Matched class-incremental comparison

The following cells execute the shared runner in `experiments/continual_comparison.py` directly. Results stay in memory; the notebook does not read generated JSON files. The synthetic 2D and Split MNIST comparisons use the same seeds, training schedule, replay budget, and conditions. Split MNIST requires the MNIST files to already be available locally under `data/mnist`; the notebook does not download data automatically.

In [ ]:
from types import SimpleNamespace
from polynomial_cl.multiclass_data import make_synthetic_2d, TASK_ORDER
from polynomial_cl.multiclass_features import polynomial_features_projected
from experiments.continual_comparison import run_experiment
from polynomial_cl.mnist import FILES

# Explicit, reproducible experiment settings. Adjust these values here when exploring.
DATA_SEED = 0
SEEDS = [0, 1, 2, 3, 4]
TRAIN_PER_CLASS = 160
TEST_PER_CLASS = 240
CLASS_SEPARATION = 2.5
CLASS_NOISE = 0.45
EPOCHS = 3
LEARNING_RATE = 0.1
BATCH_SIZE = 128
MEMORY_PER_CLASS = 200
MNIST_DIR = _REPO_ROOT / "data" / "mnist"

def experiment_args(dataset):
    return SimpleNamespace(
        dataset=dataset, data_seed=DATA_SEED,
        synthetic_train_per_class=TRAIN_PER_CLASS,
        synthetic_test_per_class=TEST_PER_CLASS,
        class_separation=CLASS_SEPARATION, class_noise=CLASS_NOISE,
        data_dir=str(MNIST_DIR), download_mnist=False, projection_seed=0, seeds=SEEDS,
        memory_per_class=MEMORY_PER_CLASS, epochs=EPOCHS,
        lr=LEARNING_RATE, batch_size=BATCH_SIZE,
    )

X_train_2d, y_train_2d, X_test_2d, y_test_2d, class_means_2d = make_synthetic_2d(
    seed=DATA_SEED, n_train_per_class=TRAIN_PER_CLASS,
    n_test_per_class=TEST_PER_CLASS, separation=CLASS_SEPARATION, noise=CLASS_NOISE,
)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for class_id in range(10):
    mask = y_train_2d == class_id
    axes[0].scatter(X_train_2d[mask, 0], X_train_2d[mask, 1], s=8, alpha=0.5, label=str(class_id))
axes[0].scatter(class_means_2d[:, 0], class_means_2d[:, 1], marker="x", s=90, linewidths=2, color="black")
axes[0].set_title("Synthetic 2D training distribution")
axes[0].set_xlabel("x1"); axes[0].set_ylabel("x2")
axes[0].legend(title="Class", ncol=2, fontsize=8)
for task, classes in enumerate(TASK_ORDER):
    mask = np.isin(y_train_2d, classes)
    axes[1].scatter(X_train_2d[mask, 0], X_train_2d[mask, 1], s=9, alpha=0.6,
                    label=f"Experience {task + 1}: classes {classes[0]}/{classes[1]}")
axes[1].set_title("Five experiences (two new classes each)")
axes[1].set_xlabel("x1"); axes[1].set_ylabel("x2")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Execute the synthetic experiment from source; no saved result file is read.
synthetic_result = run_experiment(experiment_args("synthetic_2d"))
print("Synthetic 2D comparison summary (computed in this notebook)")
for condition, summary in synthetic_result["summary"].items():
    print(
        f"{condition:>14}: final mean accuracy={summary['final_mean_accuracy']['mean']:.3f} "
        f"+/- {summary['final_mean_accuracy']['std']:.3f}; "
        f"mean forgetting={summary['mean_forgetting']['mean']:.3f} "
        f"+/- {summary['mean_forgetting']['std']:.3f}"
    )

fig, ax = plt.subplots(figsize=(8, 4))
for condition in ("fixed_linear", "expanding", "fixed_capacity"):
    selected = next(row for row in synthetic_result["runs"]
                    if row["seed"] == 0 and row["condition"] == condition)
    final_row = np.asarray([np.nan if value is None else value
                            for value in selected["accuracy_matrix"][-1]])
    ax.plot(np.arange(1, 6), final_row, marker="o", label=condition)
ax.set_xticks(np.arange(1, 6)); ax.set_xlabel("Task learned")
ax.set_ylabel("Accuracy on each task after task 5")
ax.set_title("Synthetic 2D: final task-retention profile (seed 0)")
ax.set_ylim(0, 1); ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
selected_synthetic = next(row for row in synthetic_result["runs"]
                          if row["seed"] == 0 and row["condition"] == "expanding")
degree = selected_synthetic["active_degree_by_task"][-1]
weights = np.asarray(selected_synthetic["final_weights"])
lo = X_test_2d.min(axis=0) - 0.6; hi = X_test_2d.max(axis=0) + 0.6
gx, gy = np.meshgrid(np.linspace(lo[0], hi[0], 220), np.linspace(lo[1], hi[1], 220))
grid = np.column_stack((gx.ravel(), gy.ravel())).astype(np.float32)
grid_features = polynomial_features_projected(grid, degree)
predicted = (grid_features @ weights).argmax(axis=1).reshape(gx.shape)
# Equal isotropic Gaussian classes have nearest-mean Bayes regions.
ground_truth = np.sum((grid[:, None, :] - class_means_2d[None, :, :]) ** 2, axis=2).argmin(axis=1).reshape(gx.shape)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].contourf(gx, gy, ground_truth, levels=np.arange(11) - 0.5, alpha=0.35)
axes[0].scatter(X_test_2d[:, 0], X_test_2d[:, 1], c=y_test_2d, s=4, alpha=0.35)
axes[0].set_title("Synthetic generator reference regions")
axes[1].contourf(gx, gy, predicted, levels=np.arange(11) - 0.5, alpha=0.35)
axes[1].scatter(X_test_2d[:, 0], X_test_2d[:, 1], c=y_test_2d, s=4, alpha=0.35)
axes[1].set_title("Learned expanding-model regions (seed 0)")
for ax in axes: ax.set_xlabel("x1"); ax.set_ylabel("x2")
plt.tight_layout(); plt.show()

In [ ]:
# Split MNIST is computed from local input data, not loaded from results/*.json.
# Explicitly require every archive and pass download_mnist=False to the shared loader.
missing_mnist_files = [
    filename for filename in FILES.values() if not (MNIST_DIR / filename).is_file()
]
if missing_mnist_files:
    missing = "\n".join(f"  - {filename}" for filename in missing_mnist_files)
    raise FileNotFoundError(
        f"MNIST data is incomplete under {MNIST_DIR}. Missing files:\n{missing}\n"
        "Place all four MNIST gzip files there. This notebook never downloads data."
    )
split_mnist_result = run_experiment(experiment_args("split_mnist"))
comparison_results = {"Synthetic 2D": synthetic_result, "Split MNIST": split_mnist_result}

metrics_to_plot = ("final_mean_accuracy", "final_current_accuracy",
                   "mean_forgetting", "mean_forgetting_all_classes_eligible")
for metric in metrics_to_plot:
    fig, ax = plt.subplots(figsize=(9, 4))
    labels, means, errors = [], [], []
    for dataset_name, result in comparison_results.items():
        for condition, summary in result["summary"].items():
            labels.append(f"{dataset_name}\n{condition}")
            means.append(summary[metric]["mean"]); errors.append(summary[metric]["std"])
    ax.bar(np.arange(len(labels)), means, yerr=errors, capsize=3)
    ax.set_xticks(np.arange(len(labels)), labels, rotation=25, ha="right")
    ax.set_ylim(0, max(1.0, float(np.max(np.asarray(means) + np.asarray(errors))) * 1.15))
    ax.set_ylabel(metric.replace("_", " ")); ax.set_title(f"Across-seed summary: {metric}")
    plt.tight_layout(); plt.show()

for dataset_name, result in comparison_results.items():
    selected = next(row for row in result["runs"]
                    if row["seed"] == 0 and row["condition"] == "expanding")
    matrix = np.asarray([[np.nan if value is None else value for value in row]
                         for row in selected["accuracy_matrix"]], dtype=float)
    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(matrix, vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(5), [f"Exp {i + 1}" for i in range(5)])
    ax.set_yticks(range(5), [f"Task {i + 1}" for i in range(5)])
    ax.set_xlabel("Evaluation after experience"); ax.set_ylabel("Previously introduced task")
    ax.set_title(f"{dataset_name}: expanding, seed 0")
    fig.colorbar(image, ax=ax, label="Accuracy"); plt.tight_layout(); plt.show()

# Prediction-level masking diagnostics compare decisions on identical examples and logits.
for dataset_name, result in comparison_results.items():
    selected = next(row for row in result["runs"]
                    if row["seed"] == 0 and row["condition"] == "expanding")
    diagnostics = selected["masking_diagnostics_by_experience"]
    print(f"\\n{dataset_name}: masking effects (expanding, seed 0)")
    for item in diagnostics:
        print(f"  exp {item['experience']}: changed={item['n_predictions_changed']}/"
              f"{item['n_test_examples']} ({item['prediction_change_rate']:.1%}); "
              f"accuracy masked={item['accuracy_masked']:.3f}, "
              f"unmasked={item['accuracy_unmasked']:.3f}")
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot([item["experience"] for item in diagnostics],
            [item["prediction_change_rate"] for item in diagnostics], marker="o")
    ax.set_xticks(np.arange(1, 6)); ax.set_ylim(0, 1)
    ax.set_xlabel("Experience"); ax.set_ylabel("Prediction change rate")
    ax.set_title(f"Effect of masking unseen classes: {dataset_name}, expanding seed 0")
    plt.tight_layout(); plt.show()

### Synthetic 2D confusion matrix

This matrix is computed from the final test predictions of the expanding condition for seed 0, using the synthetic experiment result created in this notebook. Rows are true classes; columns are predicted classes.

In [ ]:
selected = next(row for row in synthetic_result["runs"]
                if row["seed"] == 0 and row["condition"] == "expanding")
confusion = np.asarray(selected["final_confusion_matrix"])
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(confusion, cmap="Blues")
ax.set_xticks(range(10), range(10)); ax.set_yticks(range(10), range(10))
ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
ax.set_title("Synthetic 2D final confusion matrix (expanding, seed 0)")
fig.colorbar(image, ax=ax, label="Test examples")
plt.tight_layout(); plt.show()

### Interpretation and limitations

Compare means and standard deviations across paired seeds; do not treat seed 0 or a single decision-region plot as conclusive. The primary evaluation masks unseen output classes and records task accuracy after every experience. The additional diagnostic measures prediction changes on the same examples before and after masking, not merely differences in aggregate accuracy. The notebook computes results from the current source code and configuration; generated JSON outputs from CLI runs are optional local artifacts, not notebook inputs. Synthetic Gaussian clusters and projected MNIST have different input distributions, so similar metric trends do not prove their representations or failure mechanisms are identical.